In [ ]:
import sys
from pathlib import Path

print("Current dir:", Path.cwd())
sys.path.append(str(Path.cwd().parent))

import config as config
print("Loaded from:", config.__file__)

Setting up session and reading data from cloud(s3)

In [ ]:
from config import get_spark_session, s3_path, BUCKET_NAME
from pyspark.sql.functions import*
spark = get_spark_session("bronze-to-silver")

product_cat = spark.read.csv(
    s3_path("bronze", "category_translation", "product_category_name_translation.csv"),
    header=True,
    inferSchema=True
)

product_cat.show(5)

Data profiling

In [ ]:
product_cat.printSchema()

product_cat.count()

product_cat.show(10, truncate=False)

product_cat.describe().show()

from pyspark.sql.functions import col, count, when

product_cat.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in product_cat.columns
]).show()

product_cat.groupBy("product_category_name") \
    .count() \
    .filter(col("count") > 1) \
    .show()

product_cat.groupBy("product_category_name_english") \
    .count() \
    .filter(col("count") > 1) \
    .show()